Load And Use Finetuned Model

In [ ]:
# 打印关键依赖版本，便于复现环境
from importlib.metadata import version

pkgs = [
    "tiktoken",    # Tokenizer（GPT-2 的 BPE 分词器）
    "torch",       # Deep learning library
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
from pathlib import Path

# 检查第6章分类微调保存的权重文件是否存在；不存在则提示先运行 ch06.ipynb 生成
finetuned_model_path = Path("review_classifier.pth")
if not finetuned_model_path.exists():
    print(
        f"Could not find '{finetuned_model_path}'.\n"
        "Run the `ch06.ipynb` notebook to finetune and save the finetuned model."
    )

In [ ]:
from previous_chapters import GPTModel  # 复用前几章实现的 GPT 模型类


# GPT-2 通用基础配置
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size（词表大小）
    "context_length": 1024,  # Context length（最大上下文长度）
    "drop_rate": 0.0,        # Dropout rate（推理设 0）
    "qkv_bias": True         # Query-key-value bias（匹配 OpenAI 权重需为 True）
}

# 各规模 GPT-2 的结构差异（嵌入维度/层数/头数）
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-small (124M)"  # 第6章分类用的是 124M 小模型

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])  # 把所选规模参数并入基础配置

# Initialize base model
model = GPTModel(BASE_CONFIG)  # 先按语言模型结构实例化（下一步再换成分类头）

In [ ]:
import torch

# Convert model to classifier as in section 6.5 in ch06.ipynb
# 按书中 6.5 节把语言模型头换成二分类头：输出维度从 vocab_size 改为 num_classes=2
num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

# Then load pretrained weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 有 GPU 用 GPU
# 载入微调好的分类器权重（结构须与保存时一致）；weights_only=True 更安全
model.load_state_dict(torch.load("review_classifier.pth", map_location=device, weights_only=True))
model.to(device)
model.eval();  # 切评估模式；行尾分号抑制 Jupyter 输出
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
# This function was implemented in ch06.ipynb
# 对单条文本做垃圾/正常分类（与 ch06.ipynb 中实现一致）
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # Prepare inputs to the model
    input_ids = tokenizer.encode(text)                       # 文本 -> token id 列表
    supported_context_length = model.pos_emb.weight.shape[0] # 模型支持的最大位置数

    # Truncate sequences if they too long
    # 截断到 min(max_length, 模型上下文上限)。注意：本函数所有调用都显式传了 max_length=120，
    # 因此这里不会触发 min(None, int)；若以 max_length=None 调用则会报 TypeError（本 notebook 不涉及）
    input_ids = input_ids[:min(max_length, supported_context_length)]

    # Pad sequences to the longest sequence
    input_ids += [pad_token_id] * (max_length - len(input_ids))  # 用 pad token 补齐到 max_length
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # add batch dimension（加 batch 维 -> (1, max_length)）

    # Model inference
    with torch.no_grad():  # 推理不需要梯度，省显存
        logits = model(input_tensor.to(device))[:, -1, :]  # Logits of the last output token（取最后一个时间步的分类 logits，形状 (1, 2)）
    predicted_label = torch.argmax(logits, dim=-1).item()  # 取概率最大的类别下标

    # Return the classified result
    return "spam" if predicted_label == 1 else "not spam"  # 1=垃圾，0=正常
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

# 一条典型的垃圾短信，预期分类为 spam
print(classify_review(
    text_1, model, tokenizer, device, max_length=120
))

In [ ]:
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)

# 一条正常的日常短信，预期分类为 not spam
print(classify_review(
    text_2, model, tokenizer, device, max_length=120
))